# Bước 2 & Bước 3: Xây dựng Mô hình GCN Đầu tiên & Bộ Đánh giá Dùng chung

**Luồng xử lý chuẩn:**
`Đồ thị đầu vào (x, edge_index) → GCNEncoder → Vector từng nút (z) → LinkDecoder(z_u, z_v) → Điểm liên kết (Logit)`

**Các bước thực hiện trong Notebook này:**
1. Khởi tạo `GCNEncoder` (2 lớp `GCNConv` + chuẩn hóa + kết nối tắt) và `LinkDecoder` nhận biểu diễn 2 nút trả về 1 logit.
2. Huấn luyện với `BCEWithLogitsLoss`, theo dõi `Validation AP` qua từng epoch và lưu checkpoint tốt nhất (`outputs/checkpoints/gcn_baseline_best.pt`).
3. Nạp lại checkpoint tốt nhất, dùng `torch.sigmoid` chuyển logit thành điểm dự đoán trong khoảng `[0, 1]`.
4. Chạy **Bộ đánh giá dùng chung (`src/metrics.py`)** tính `ROC-AUC`, `AP`, `F1` (chọn ngưỡng trên Validation, giữ cố định trên Test) và `Hits@10`.

## 2.1. Nạp dữ liệu đồ thị đồng nhất và các tập Train / Val / Test

In [1]:
import os
import sys
import torch
import pandas as pd

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_validator import build_homogeneous_customer_graph, load_split_pairs
from src.models import GNNLinkPredictor, train_and_evaluate_model
from src.metrics import evaluate_all, evaluate_by_negative_types

graph_dict = build_homogeneous_customer_graph()
splits = load_split_pairs()
x = graph_dict["x"]
edge_index = graph_dict["edge_index"]

print(f"Loaded graph: x={tuple(x.shape)}, edge_index={tuple(edge_index.shape)}")

Loaded graph: x=(50000, 60), edge_index=(2, 557062)


## 2.2. Kiểm tra kiến trúc `GCNEncoder` + `LinkDecoder` & Huấn luyện lưu Checkpoint theo Validation AP

In [2]:
ckpt_path = os.path.join(PROJECT_ROOT, "outputs", "checkpoints", "gcn_baseline_best.pt")

gcn_run = train_and_evaluate_model(
    model_type="GCN",
    x=x,
    edge_index=edge_index,
    splits=splits,
    seed=42,
    epochs=60,
    lr=3e-3,
    hidden_channels=64,
    out_channels=64,
    dropout=0.2,
    checkpoint_path=ckpt_path,
    verbose=True,
)

print("\nKiến trúc mô hình GCN:")
print(gcn_run["model"])
print(f"\nCheckpoint tốt nhất được lưu tại epoch {gcn_run['best_epoch']} với Val AP = {gcn_run['best_val_ap']:.4f}")

[GCN | Seed 42] Epoch 001/060 | Train Loss: 0.6975 | Val Loss: 0.6472 | Val AP: 0.7637 (Best: 0.7637 @ Ep 1)
[GCN | Seed 42] Epoch 020/060 | Train Loss: 0.4822 | Val Loss: 0.5082 | Val AP: 0.7853 (Best: 0.7853 @ Ep 20)
[GCN | Seed 42] Epoch 040/060 | Train Loss: 0.3878 | Val Loss: 0.5092 | Val AP: 0.8405 (Best: 0.8417 @ Ep 39)
[GCN | Seed 42] Epoch 060/060 | Train Loss: 0.3064 | Val Loss: 0.5691 | Val AP: 0.8492 (Best: 0.8503 @ Ep 50)

Kiến trúc mô hình GCN:
GNNLinkPredictor(
  (encoder): GCNEncoder(
    (conv1): GCNConv(
      (lin): Linear(in_features=60, out_features=64, bias=False)
    )
    (bn1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): GCNConv(
      (lin): Linear(in_features=64, out_features=64, bias=False)
    )
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (skip): Linear(in_features=60, out_features=64, bias=True)
  )
  (decoder): LinkDecoder(
    (mlp): Sequential(
      (0):

## 2.3. Kiểm tra Nạp lại Checkpoint & Xuất Điểm Dự đoán Validation / Test

In [3]:
loaded_ckpt = torch.load(ckpt_path, weights_only=False)
device = next(gcn_run["model"].parameters()).device

reloaded_model = GNNLinkPredictor(**loaded_ckpt["config"]).to(device)
reloaded_model.load_state_dict(loaded_ckpt["model_state_dict"])

val_probs = reloaded_model.predict_proba(
    x.to(device), edge_index.to(device), splits["val"]["edge_label_index"].to(device)
).cpu().numpy()
test_probs = reloaded_model.predict_proba(
    x.to(device), edge_index.to(device), splits["test"]["edge_label_index"].to(device)
).cpu().numpy()

print("5 điểm dự đoán đầu tiên trên tập Validation (sau Sigmoid, thuộc [0, 1]):", val_probs[:5].round(4))
print("Kiểm tra khớp hoàn toàn với kết quả trước khi lưu:", bool(abs(val_probs - gcn_run["val_y_score"]).max() < 1e-6))

5 điểm dự đoán đầu tiên trên tập Validation (sau Sigmoid, thuộc [0, 1]): [0.4541 0.8648 0.7954 0.8326 0.138 ]
Kiểm tra khớp hoàn toàn với kết quả trước khi lưu: True


## 3.1. Chạy Bộ Đánh giá Dùng chung (`src/metrics.py`) cho GCN

- Bộ đánh giá nhận đầu vào độc lập gồm `(val_y_true, val_y_score, test_y_true, test_y_score)` — bất kỳ thành viên nào trong nhóm cũng có thể truyền điểm dự đoán của mô hình khác vào hàm `evaluate_all`.
- Ngưỡng `F1` được chọn trên tập Validation và giữ nguyên khi chấm trên tập Test.

In [4]:
gcn_overall_row = evaluate_all(
    val_y_true=gcn_run["val_y_true"],
    val_y_score=val_probs,
    test_y_true=gcn_run["test_y_true"],
    test_y_score=test_probs,
    scenario_name="Overall",
    model_name="GCN",
    seed=42,
)

print("=== DÒNG KẾT QUẢ CHUẨN CỦA GCN (BƯỚC 3) ===")
display(pd.DataFrame([gcn_overall_row]))

print("\n=== ĐÁNH GIÁ CHI TIẾT GCN TRÊN TỪNG KỊCH BẢN MẪU ÂM (SCENARIO BREAKDOWN) ===")
gcn_scenario_df = evaluate_by_negative_types(
    val_y_true=gcn_run["val_y_true"],
    val_y_score=val_probs,
    test_y_true=gcn_run["test_y_true"],
    test_y_score=test_probs,
    test_edge_type=splits["test"]["edge_type"],
    model_name="GCN",
    seed=42,
)
display(gcn_scenario_df)

=== DÒNG KẾT QUẢ CHUẨN CỦA GCN (BƯỚC 3) ===

=== ĐÁNH GIÁ CHI TIẾT GCN TRÊN TỪNG KỊCH BẢN MẪU ÂM (SCENARIO BREAKDOWN) ===


Scenario,Model,Seed,ROC-AUC,AP,F1,Hits@10,Val_ROC-AUC,Val_AP,Val_F1,Val_Hits@10,Best_Threshold,Test_Precision,Test_Recall
Overall,GCN,42,0.8151,0.8057,0.7576,0.6807,0.8572,0.8503,0.8033,0.7611,0.2455,0.7111,0.8105


Scenario,Model,Seed,ROC-AUC,AP,F1,Hits@10,Threshold,Num_Pos,Num_Neg
Overall (All 5 Neg Types),GCN,42,0.8151,0.8057,0.7576,0.6807,0.2455,1710,1710
Scenario 1: AML-AML (Cross-Case),GCN,42,0.6009,0.8428,0.7943,0.3474,0.2455,1710,513
Scenario 2: AML-Clean,GCN,42,0.8536,0.9771,0.8712,0.7550,0.2455,1710,256
Scenario 3: Suspected KYC,GCN,42,0.9195,0.9872,0.8870,0.8585,0.2455,1710,256
Scenario 4: Shared IP/Device,GCN,42,0.8388,0.9944,0.8930,0.9170,0.2455,1710,32
Scenario 5: Clean-Clean Random,GCN,42,0.9263,0.9745,0.8822,0.8772,0.2455,1710,653
